# RSA from scratch
How does RSA compute with 2048-bit numbers, and why does a private key store more than $(N, e, d)$?
We compute RSA encryption and decryption ourselves with the key from `02` and cross-check with OpenSSL in both directions.

Run `02_openssl_keys.ipynb` first (it creates the key files) and complete the grammar `RSA_pub.asn` in `04_asn1tools_rsa_math.ipynb` –
this notebook decodes the key with it and then computes with the decoded values. Run cells from top to bottom. Modify the cells marked `TODO` (the check cells after them verify your implementation).

Written answers (**Your answer** cells) are not checked automatically – after you finish a task, verify them against the notebook in `solutions/`.

In [ ]:
import os
import secrets
import time

for name in ["RSA_2048_priv.der", "RSA_2048_priv.pem", "RSA_2048_pub.pem"]:
    assert os.path.exists(name), f"{name} is missing - run 02_openssl_keys.ipynb first"

## Part 1: How modular exponentiation works
RSA computes $x^y \bmod m$ with 2048-bit numbers. Multiplying by $x$ exactly $y$ times would take about $2^{2048}$ steps – impossible.
**Square-and-multiply** starts with $x$ for the leading bit of $y$ and goes through the remaining bits: square the result in every step, and multiply by $x$ where the bit is 1.
Every intermediate result is reduced modulo $m$, so the numbers never grow beyond $m^2$.
Example: $a^{11}$, $11 = 1011_2$: $a \to a^2 \to a^4 \cdot a = a^5 \to a^{10} \cdot a = a^{11}$ – about $2 \log_2 y$ multiplications.
(The slides compute the same power from the lowest bit: $a^{11} = a^1 \cdot a^2 \cdot a^8$.)
Python's `pow(x, y, m)` does the same in C. `pow_sm` below returns a pair `(value, number_of_multiplications)`.

In [ ]:
def pow_naive(x, y, m):
    r, steps = 1, 0
    for _ in range(y):
        r = r * x % m
        steps += 1
    return r, steps

def pow_sm(x, y, m):
    """x^y mod m (y >= 1) and the number of multiplications."""
    bits = bin(y)[2:]
    r, steps = x % m, 0        # the leading bit is 1
    for bit in bits[1:]:
        r = r * r % m          # square
        steps += 1
        if bit == "1":
            r = r * x % m      # multiply
            steps += 1
    return r, steps

x, y, m = 7, 100_000, 1_000_003
t0 = time.perf_counter(); r1, s1 = pow_naive(x, y, m)
t1 = time.perf_counter(); r2, s2 = pow_sm(x, y, m)
t2 = time.perf_counter()
assert r1 == r2 == pow(x, y, m)
print(f"naive:               {s1:7} multiplications, {t1 - t0:.4f} s")
print(f"square-and-multiply: {s2:7} multiplications, {t2 - t1:.6f} s")

y = secrets.randbits(2048)
print(f"2048-bit exponent: square-and-multiply needs {pow_sm(x, y, m)[1]} multiplications, the naive loop about 2^2048")

The modular inverse $b^{-1} \bmod m$ (computed by the extended Euclidean algorithm) is `pow(b, -1, m)`; it exists only if $\gcd(b, m) = 1$.

In [ ]:
print(pow(3, -1, 7), 3 * pow(3, -1, 7) % 7)  # slides: 3^-1 mod 7 = 5
# pow(4, -1, 10)  # ValueError: base is not invertible - gcd(4, 10) = 2

## Part 2: RSA with the key from `02`
RSA: encryption $c = m^e \bmod N$, decryption $m = c^d \bmod N$.
Before encryption, OpenSSL pads the message into a 256-byte block (PKCS#1 v1.5). Padding has a security purpose: its random bytes make
every encryption of the same message different, and it turns even a short message into a large number, so attacks on small or structured
messages do not work (see the optional part at the end). We treat padding as a black box – `pad` and `unpad` are given, the tasks are only about the RSA computation.

In [ ]:
import asn1tools

rsa_spec = asn1tools.compile_files("RSA_pub.asn")  # your grammar from 04
key = rsa_spec.decode("RSAPrivateKey", open("RSA_2048_priv.der", "rb").read())
N, e, d = key["modulus"], key["publicExponent"], key["privateExponent"]

# padding (PKCS#1 v1.5) as a black box: message bytes <-> number smaller than N
def pad(message, k=256):
    ps_len = k - 3 - len(message)
    assert ps_len >= 8, "message too long: PKCS#1 v1.5 needs at least 8 padding bytes"
    ps = bytes(secrets.randbelow(255) + 1 for _ in range(ps_len))  # random non-zero bytes
    return int.from_bytes(b"\x00\x02" + ps + b"\x00" + message, "big")

def unpad(m, k=256):
    block = m.to_bytes(k, "big")
    assert block[:2] == b"\x00\x02", "not a PKCS#1 v1.5 block"
    return block[block.index(b"\x00", 2) + 1:]

### Task 1: OpenSSL encrypts, you decrypt
Decrypt with your `pow_sm`; the check compares with the built-in `pow` and removes the padding.

In [ ]:
open("PT.txt", "w").write("Hello PV181")
!openssl pkeyutl -encrypt -pubin -inkey RSA_2048_pub.pem -in PT.txt -out RSA_CT.bin
c = int.from_bytes(open("RSA_CT.bin", "rb").read(), "big")

In [ ]:
m = ...  # TODO: decrypt c with pow_sm (it returns (value, steps) - use [0])

In [ ]:
assert m == pow(c, d, N)
print(unpad(m))
assert unpad(m) == b"Hello PV181"

**a)** Which key values does decryption use, and why is `unpad` needed?

**Your answer** (double-click to edit)

**a)** …

### Task 2: You encrypt, OpenSSL decrypts
Pad a message and encrypt it with your `pow_sm` and the public key; the check writes the ciphertext to a file and lets OpenSSL decrypt it.

In [ ]:
m = pad(b"Hello from Python")
c = ...  # TODO: encrypt m with pow_sm (use [0])

In [ ]:
assert c == pow(m, e, N)
open("my_CT.bin", "wb").write(c.to_bytes(256, "big"))
!openssl pkeyutl -decrypt -inkey RSA_2048_priv.pem -in my_CT.bin -out my_PT.txt
print(open("my_PT.txt", "rb").read())
assert open("my_PT.txt", "rb").read() == b"Hello from Python"

**a)** Why can OpenSSL decrypt a ciphertext computed in Python?

**Your answer** (double-click to edit)

**a)** …

### Task 3: Why the key stores `exponent1`, `exponent2` and `coefficient`
They allow decryption with the Chinese remainder theorem (CRT): two exponentiations with half-size numbers instead of one with the full $N$:

$m_1 = c^{d_P} \bmod p, \quad m_2 = c^{d_Q} \bmod q, \quad h = q_{inv} \cdot (m_1 - m_2) \bmod p, \quad m = m_2 + h \cdot q$

with $d_P$ = `exponent1`, $d_Q$ = `exponent2`, $q_{inv}$ = `coefficient` (it combines the two partial results into $m$).
CRT needs the secret primes, so only the private-key owner can use it. Decrypt the ciphertext from Task 1 with CRT and compare the speed (times are approximate and depend on the machine).

In [ ]:
p, q = key["prime1"], key["prime2"]
dP, dQ, qInv = key["exponent1"], key["exponent2"], key["coefficient"]
c = int.from_bytes(open("RSA_CT.bin", "rb").read(), "big")

def decrypt_crt(c):
    m1 = ...  # TODO: with pow_sm
    m2 = ...  # TODO: with pow_sm
    h = ...   # TODO
    return m2 + h * q

In [ ]:
assert decrypt_crt(c) == pow(c, d, N)

t0 = time.perf_counter()
for _ in range(20):
    pow_sm(c, d, N)
t1 = time.perf_counter()
for _ in range(20):
    decrypt_crt(c)
t2 = time.perf_counter()
print(f"20 decryptions without CRT: {t1 - t0:.2f} s, with CRT: {t2 - t1:.2f} s, speed-up about {(t1 - t0) / (t2 - t1):.1f}x")

**a)** Why is decryption with CRT faster, and what speed-up do you expect in theory? Compare it with the measured one. Hint: the cost of one multiplication grows roughly with the square of the length of the numbers.

**Your answer** (double-click to edit)

**a)** …

### Optional: why padding matters with $e = 3$
Without padding, a short message $m$ with $m^3 < N$ is never reduced modulo $N$: $c = m^3$ is an ordinary integer and $m$ is its ordinary cube root.
This works only if the message is unpadded, small enough that $m^3 < N$, and the attacker knows $c$ – padding makes every $m$ nearly as large as $N$.

In [ ]:
def icbrt(x):
    """Integer cube root by binary search."""
    lo, hi = 0, 1 << (x.bit_length() // 3 + 1)
    while lo < hi:
        mid = (lo + hi) // 2
        if mid ** 3 < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

m = int.from_bytes(b"secret PIN 1234", "big")
c = pow(m, e, N)  # textbook RSA without padding, e = 3
print("c == m^3 without reduction:", c == m ** 3)
print(icbrt(c).to_bytes(15, "big"))